# Session 4: PID 制御の理論と実装
# Session 4: PID Control Theory and Implementation

## 目標 / Objective

P/I/D 各項の役割を理解し、工学形式 (Kp, Ti, Td) で PID を設計する。

Understand the role of P/I/D terms and design PID controllers in engineering form (Kp, Ti, Td).

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| PID の伝達関数 | $C(s) = K_p(1 + \frac{1}{T_i s} + T_d s)$ |
| 各項の役割 | P: 即応性、I: 定常偏差除去、D: 安定性向上 |
| 不完全微分 | $\frac{T_d s}{\eta T_d s + 1}$ による高周波ノイズ抑制 |
| ボード線図 | 周波数応答による PID 設計 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stampfly_edu.sim import (
    FirstOrderPlant, SecondOrderPlant,
    simulate_pid, compare_controllers,
)
from stampfly_edu.sim.simulate import compute_metrics

print("Ready! / 準備完了！")

## 1. PID 伝達関数 / PID Transfer Function

### 工学形式 (Engineering Form)

$$C(s) = K_p \left(1 + \frac{1}{T_i s} + \frac{T_d s}{\eta T_d s + 1}\right)$$

| パラメータ | 名称 | 役割 |
|-----------|------|------|
| $K_p$ | 比例ゲイン | 応答の速さ |
| $T_i$ | 積分時間 [s] | 定常偏差の除去速度 |
| $T_d$ | 微分時間 [s] | 変化への先読み |
| $\eta$ | 微分フィルタ係数 | 高周波ノイズ抑制 (0.1〜0.2) |

### StampFly ファームウェアとの対応

StampFly のファームウェアはこの工学形式を直接使用しています。
`simulator/vpython/control/pid.py` の PID クラスも同じ実装です。

## 2. P/I/D 各項の効果 / Effect of P, I, D Terms

In [ ]:
# Effect of each PID term
# PID 各項の効果
plant = SecondOrderPlant(wn=5.0, zeta=0.3)  # Under-damped plant

configs = [
    {"Kp": 2.0, "Ti": 0.0, "Td": 0.0},   # P only
    {"Kp": 2.0, "Ti": 1.0, "Td": 0.0},   # PI
    {"Kp": 2.0, "Ti": 0.0, "Td": 0.2},   # PD
    {"Kp": 2.0, "Ti": 1.0, "Td": 0.2},   # PID
]
labels = ["P only", "PI", "PD", "PID"]

fig = compare_controllers(
    plant, configs, labels=labels,
    title="P / PI / PD / PID Comparison on 2nd-Order Plant\n2次系での比較",
    duration=8.0,
)
plt.show()

## 3. 微分項と不完全微分フィルタ / Derivative and Incomplete Derivative Filter

### なぜフィルタが必要か？

理想微分 $T_d s$ は高周波ノイズを無限に増幅します。
不完全微分フィルタで帯域を制限：

$$D(s) = \frac{T_d s}{\eta T_d s + 1}$$

カットオフ周波数: $f_c = \frac{1}{2\pi \eta T_d}$

In [ ]:
# Effect of derivative filter coefficient eta
# 微分フィルタ係数 eta の効果
plant = SecondOrderPlant(wn=5.0, zeta=0.3)

configs = [
    {"Kp": 2.0, "Ti": 1.0, "Td": 0.2, "eta": 0.01},   # Almost ideal
    {"Kp": 2.0, "Ti": 1.0, "Td": 0.2, "eta": 0.125},  # Default
    {"Kp": 2.0, "Ti": 1.0, "Td": 0.2, "eta": 0.5},    # Heavy filter
]
labels = ["eta=0.01 (sharp)", "eta=0.125 (default)", "eta=0.5 (smooth)"]

fig = compare_controllers(
    plant, configs, labels=labels,
    title="Derivative Filter Effect / 微分フィルタの効果",
    duration=8.0,
)
plt.show()

## 4. ボード線図によるPID設計 / PID Design with Bode Plot

周波数応答でPIDの効果を見てみましょう。

In [ ]:
# Bode plot of PID controller
# PID コントローラのボード線図
from scipy import signal

Kp, Ti, Td, eta = 2.0, 1.0, 0.2, 0.125

# PID transfer function: Kp(1 + 1/(Ti*s) + Td*s/(eta*Td*s + 1))
# Combine into single transfer function
# P term
P_num = [Kp]
P_den = [1]

# Build open-loop with plant G(s) = wn^2/(s^2 + 2*zeta*wn*s + wn^2)
wn, zeta = 5.0, 0.3

# Frequency range
w = np.logspace(-2, 3, 1000)
s = 1j * w

# Plant
G = wn**2 / (s**2 + 2*zeta*wn*s + wn**2)

# PID controller
C = Kp * (1 + 1/(Ti*s) + Td*s/(eta*Td*s + 1))

# Open-loop L = C*G
L = C * G

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Magnitude
ax1.semilogx(w, 20*np.log10(np.abs(G)), 'b--', label='Plant G(s)')
ax1.semilogx(w, 20*np.log10(np.abs(C)), 'g--', label='PID C(s)')
ax1.semilogx(w, 20*np.log10(np.abs(L)), 'r-', linewidth=2, label='Open-loop L(s)=C*G')
ax1.axhline(y=0, color='k', linestyle=':', linewidth=0.5)
ax1.set_ylabel('Magnitude (dB) / ゲイン')
ax1.set_title('Bode Plot / ボード線図')
ax1.legend()
ax1.grid(True, alpha=0.3, which='both')
ax1.set_ylim([-60, 40])

# Phase
ax2.semilogx(w, np.degrees(np.angle(G)), 'b--', label='Plant G(s)')
ax2.semilogx(w, np.degrees(np.angle(C)), 'g--', label='PID C(s)')
ax2.semilogx(w, np.degrees(np.angle(L)), 'r-', linewidth=2, label='Open-loop L(s)')
ax2.axhline(y=-180, color='k', linestyle=':', linewidth=0.5)
ax2.set_ylabel('Phase (deg) / 位相')
ax2.set_xlabel('Frequency (rad/s) / 周波数')
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Gain and phase margins
# ゲイン余裕と位相余裕
gain_crossover_idx = np.argmin(np.abs(20*np.log10(np.abs(L))))
phase_at_gc = np.degrees(np.angle(L[gain_crossover_idx]))
phase_margin = 180 + phase_at_gc

print(f"Gain crossover frequency / ゲイン交差周波数: {w[gain_crossover_idx]:.2f} rad/s")
print(f"Phase margin / 位相余裕: {phase_margin:.1f} deg")

## 5. インタラクティブ PID チューニング / Interactive PID Tuning

スライダーでゲインを変えながらステップ応答をリアルタイムで観察します。

（ipywidgets が必要です。Jupyter Lab/Notebook で実行してください）

In [ ]:
from stampfly_edu.widgets import pid_tuner_widget
from stampfly_edu.sim import SecondOrderPlant

plant = SecondOrderPlant(wn=5.0, zeta=0.3)
pid_tuner_widget(plant, duration=8.0)

## 6. 考察課題 / Discussion Questions

1. **ゲイン余裕と位相余裕**: 位相余裕が小さいとステップ応答にどう影響するか？
2. **P, I, D の優先順位**: PID 調整でまず何から合わせるべきか？その理由は？
3. **サンプリング周期の影響**: 制御周期が遅くなると PID の各項にどう影響するか？

---

1. **Gain & Phase Margins**: How does a small phase margin affect step response?
2. **P, I, D Priority**: Which term should you tune first? Why?
3. **Sampling Period**: How does a slower control rate affect each PID term?

In [ ]:
# Your experiments here / ここで実験してみてください
